## Quack Elo Prototype Notebook

This is intended to be a prototype notebook. It is suposed to be implemented as a module / service which calculates a customized version of elo per player after each map, by accessing the database.

In [30]:
import pandas as pd
from db.connection_db import get_conn


def table_columns(conn, table_name: str) -> set[str]:
    rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    return {str(r[1]) for r in rows}


def first_existing(columns: set[str], candidates: list[str], label: str) -> str:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise RuntimeError(f"Could not find a usable {label} column. Tried: {candidates}")


with get_conn() as conn:
    mps_cols = table_columns(conn, "match_player_stats")
    m_cols = table_columns(conn, "matches")

    steam_col = first_existing(mps_cols, ["steamid64", "steam64_id"], "steam id")
    name_col = first_existing(mps_cols, ["name", "player_name"], "player name")
    team_col = first_existing(mps_cols, ["team", "team_name"], "team")

    # 1) matchhistory table (match_id as unique key) sorted by date

    if "match_id" in m_cols:
        preferred_cols = [
            "match_id",
            "match_time",
            "start_time",
            "end_time",
            "created_at",
            "finished_at",
            "winner",
            "team1_score",
            "team2_score",
        ]
        match_cols = [c for c in preferred_cols if c in m_cols]
        matchhistory_df = pd.read_sql_query(
            f"SELECT {', '.join(match_cols)} FROM matches ORDER BY match_id DESC",
            conn,
        )
    else:
        matchhistory_df = pd.read_sql_query(
            """
            SELECT DISTINCT match_id
            FROM match_player_stats
            ORDER BY match_id DESC
            """,
            conn,
        )

    matchhistory_df = matchhistory_df.drop_duplicates(subset=["match_id"], keep="first")
    matchhistory_df = matchhistory_df.set_index("match_id", drop=False)
    # sort by date if possible
    date_cols = [c for c in ["match_time", "start_time", "end_time", "created_at", "finished_at"] if c in matchhistory_df.columns]
    if date_cols:
        matchhistory_df = matchhistory_df.sort_values(date_cols, ascending=False, kind="stable")

    # 2) playerlist table (teams per match)
    playerlist_df = pd.read_sql_query(
        f"""
        SELECT DISTINCT
            mps.match_id,
            CAST(mps.{steam_col} AS TEXT) AS steamid,
            TRIM(mps.{team_col}) AS team_name,
            TRIM(mps.{name_col}) AS player_name
        FROM match_player_stats mps
        WHERE mps.{steam_col} IS NOT NULL
          AND TRIM(COALESCE(mps.{team_col}, '')) <> ''
          AND TRIM(COALESCE(mps.{name_col}, '')) <> ''
        ORDER BY mps.match_id DESC, team_name, player_name
        """,
        conn,
    )
    playerlist_df = playerlist_df.drop_duplicates(
        subset=["match_id", "team_name", "steamid"],
        keep="first",
    )

    # 3) players table (distinct players)
    players_df = playerlist_df[["steamid", "player_name", "match_id"]].copy()
    players_df = players_df.sort_values(["steamid", "match_id"], ascending=[True, False])
    players_df = players_df.drop_duplicates(subset=["steamid"], keep="first")
    players_df = players_df.drop(columns=["match_id"]).rename(columns={"player_name": "name"})
    players_df = players_df.sort_values(["name", "steamid"], kind="stable").reset_index(drop=True)

print("Built independent tables:")
print(f"matchhistory_df rows={len(matchhistory_df):,} unique_match_id={matchhistory_df['match_id'].nunique():,}")
print(f"playerlist_df rows={len(playerlist_df):,} unique_match_id={playerlist_df['match_id'].nunique():,}")
print(f"players_df rows={len(players_df):,} unique_steamid={players_df['steamid'].nunique():,}")

print("\nmatchhistory_df preview:")
#display(matchhistory_df)

print("\nplayerlist_df preview:")
#display(playerlist_df)

print("\nplayers_df preview:")
#display(players_df)

Built independent tables:
matchhistory_df rows=12 unique_match_id=12
playerlist_df rows=125 unique_match_id=12
players_df rows=23 unique_steamid=23

matchhistory_df preview:

playerlist_df preview:

players_df preview:


In [31]:
# Unified player x match outcome table (win/loose matrix)

# Keep one row per player per match by taking one team assignment in that match.
player_match_base_df = (
    playerlist_df.sort_values(["match_id", "steamid", "team_name"], kind="stable")
    .drop_duplicates(subset=["match_id", "steamid"], keep="last")
    .merge(
        matchhistory_df[["match_id", "winner"]].reset_index(drop=True),
        on="match_id",
        how="left",
    )
)

# Attach canonical player names from players_df.
player_match_outcome_df = player_match_base_df.merge(
    players_df.rename(columns={"name": "player_name_canonical"}),
    on="steamid",
    how="left",
)

player_match_outcome_df["player_name"] = (
    player_match_outcome_df["player_name_canonical"]
    .fillna(player_match_outcome_df["player_name"])
)
player_match_outcome_df = player_match_outcome_df.drop(columns=["player_name_canonical"])


# Normalize match-specific team labels to TeamA / TeamB.
def _normalize_team_text(value):
    txt = str(value or "").strip()
    return txt


def _is_canonical_team(value):
    txt = _normalize_team_text(value).lower()
    return txt in {"teama", "teamb"}


def _canonical_label(value):
    txt = _normalize_team_text(value).lower()
    if txt == "teama":
        return "TeamA"
    if txt == "teamb":
        return "TeamB"
    return _normalize_team_text(value)


def _build_team_mapping_for_match(group):
    names = []
    for raw in group["team_name"].tolist():
        t = _normalize_team_text(raw)
        if not t or t.lower() == "all":
            continue
        if t not in names:
            names.append(t)

    # If already canonical labels are present, keep them untouched.
    if set(t.lower() for t in names).issubset({"teama", "teamb"}):
        return {t: _canonical_label(t) for t in names}

    # Otherwise assign the two match team names to TeamA/TeamB.
    if len(names) >= 2:
        chosen = names[:2]
        return {chosen[0]: "TeamA", chosen[1]: "TeamB"}

    if len(names) == 1:
        return {names[0]: "TeamA"}

    return {}


team_map_by_match = {
    match_id: _build_team_mapping_for_match(group)
    for match_id, group in player_match_outcome_df.groupby("match_id", sort=False)
}


def _map_team_name(match_id, team_name):
    team_raw = _normalize_team_text(team_name)
    if not team_raw:
        return team_raw
    if _is_canonical_team(team_raw):
        return _canonical_label(team_raw)

    mapping = team_map_by_match.get(match_id, {})
    if team_raw in mapping:
        return mapping[team_raw]
    return team_raw


def _map_winner_name(match_id, winner_name):
    winner_raw = _normalize_team_text(winner_name)
    if not winner_raw:
        return winner_raw
    if _is_canonical_team(winner_raw):
        return _canonical_label(winner_raw)

    mapping = team_map_by_match.get(match_id, {})
    if winner_raw in mapping:
        return mapping[winner_raw]
    return winner_raw


player_match_outcome_df["team_name"] = player_match_outcome_df.apply(
    lambda r: _map_team_name(r["match_id"], r["team_name"]), axis=1
)
player_match_outcome_df["winner"] = player_match_outcome_df.apply(
    lambda r: _map_winner_name(r["match_id"], r["winner"]), axis=1
)


def _result_label(row):
    winner = str(row.get("winner") or "").strip()
    team = str(row.get("team_name") or "").strip()
    if not winner or not team:
        return "unknown"
    if team.lower() == "all":
        return "unknown"
    return "win" if team == winner else "loose"


player_match_outcome_df["result"] = player_match_outcome_df.apply(_result_label, axis=1)

# Distinct players x Match ID matrix
player_key = (
    player_match_outcome_df["player_name"].astype(str)
    + " ["
    + player_match_outcome_df["steamid"].astype(str)
    + "]"
)

player_match_matrix_df = (
    player_match_outcome_df.assign(player=player_key)
    .pivot_table(
        index="player",
        columns="match_id",
        values="result",
        aggfunc="first",
    )
    .fillna("")
)

# Keep matches left->right as 1..N (older to newer); use numeric sort when possible.
_sorted_match_cols = sorted(
    list(player_match_matrix_df.columns),
    key=lambda v: int(v) if str(v).isdigit() else 10**12,
)
player_match_matrix_df = player_match_matrix_df[_sorted_match_cols]

print(f"Outcome rows (long format): {len(player_match_outcome_df):,}")
print(f"Matrix shape (players x matches): {player_match_matrix_df.shape[0]} x {player_match_matrix_df.shape[1]}")
print("\nplayers x match matrix:")
display(player_match_matrix_df)

Outcome rows (long format): 125
Matrix shape (players x matches): 23 x 12

players x match matrix:


match_id,1,2,3,4,5,6,7,8,9,10,11,12
player,,,,,,,,,,,,
-Zuria- [76561198010796900],,unknown,loose,,,,,,,,,
Amü [76561198962927846],win,unknown,,,,win,,,loose,unknown,loose,unknown
Captain Blake [76561198051680113],win,unknown,loose,win,loose,,loose,win,win,unknown,loose,unknown
Chicca —ฅ/ᐠ. ̫ .ᐟ\ฅ — [76561198260260482],loose,,,,,,loose,win,,,,
Couchyyy1337 [76561198034161077],win,unknown,loose,loose,win,loose,win,loose,,,loose,
Dottersack [76561198240903150],loose,unknown,loose,win,win,loose,loose,loose,loose,unknown,loose,unknown
Dude [76561197970397706],loose,unknown,win,win,,loose,win,win,,,,
Inay [76561198007257679],,,,,loose,,,,,,win,
KeTaNoS [76561198045998047],win,unknown,loose,win,win,loose,win,win,loose,unknown,win,unknown
